In [1]:
import dspy
lm = dspy.LM('ollama_chat/gemma3:4b', api_base='http://localhost:11434', api_key='')
dspy.configure(lm=lm)

/home/penhfel/miniconda3/envs/chatbot-study/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Calling the model directly

In [2]:
lm("Say this is a test!", temperature=0.7)  # => ['This is a test!']
lm(messages=[{"role": "user", "content": "Say this is a test!"}])  # => ['This is a test!']

['Okay! This is a test! 😊 \n\nLet me know if you need anything else. 👍']

Math Example

In [3]:
math = dspy.ChainOfThought("question -> answer: float")
math(question="Two dice are tossed. What is the probability that the sum equals two?")

Prediction(
    reasoning='When two dice are tossed, the possible outcomes range from 2 to 12. We want to find the probability that the sum of the two dice equals 2. The only way to achieve a sum of 2 is if both dice show a 1. There are a total of 6 * 6 = 36 possible outcomes when tossing two dice. The number of favorable outcomes (sum equals 2) is 1 (1, 1). Therefore, the probability is 1/36.',
    answer=1.0000000000000002
)

Classification

In [4]:
from typing import Literal

class Classify(dspy.Signature):
    """Classify sentiment of a given sentence."""

    sentence: str = dspy.InputField()
    sentiment: Literal['positive', 'negative', 'neutral'] = dspy.OutputField()
    confidence: float = dspy.OutputField()

classify = dspy.Predict(Classify)
classify(sentence="This book was super fun to read, though not the last chapter.")

Prediction(
    sentiment='positive',
    confidence=0.95
)

In [5]:
from typing import Literal

class Classify(dspy.Signature):
    """Classify the stance of a given sentence regarding Donald Trump.
    The stance can be support if the sentence supports Donald Trump, deny if the stance if against Donald Trump
    or neutral if the Sentence does not mention Donald Trump"""

    sentence: str = dspy.InputField()
    stance: Literal['support', 'deny', 'neutral'] = dspy.OutputField()
    confidence: float = dspy.OutputField()

classify = dspy.Predict(Classify)
classify(sentence="I am sad today, Trump won the election")

Prediction(
    stance='support',
    confidence=0.95
)

Information Extraction

In [6]:
class ExtractInfo(dspy.Signature):
    """Extract structured information from text."""

    text: str = dspy.InputField()
    title: str = dspy.OutputField()
    headings: list[str] = dspy.OutputField()
    entities: list[dict[str, str]] = dspy.OutputField(desc="a list of entities and their metadata")

module = dspy.Predict(ExtractInfo)

text = "Apple Inc. announced its latest iPhone 14 today." \
    "The CEO, Tim Cook, highlighted its new features in a press release."
response = module(text=text)

print(response.title)
print(response.headings)
print(response.entities)

Apple iPhone 14 Announcement
['Apple iPhone 14 Announcement']
[{'company': 'Apple Inc.'}, {'product': 'iPhone 14'}, {'person': 'Tim Cook'}]


Multi Stage Pipeline

In [7]:
class Outline(dspy.Signature):
    """Outline a thorough overview of a topic."""

    topic: str = dspy.InputField()
    title: str = dspy.OutputField()
    sections: list[str] = dspy.OutputField()
    section_subheadings: dict[str, list[str]] = dspy.OutputField(desc="mapping from section headings to subheadings")

class DraftSection(dspy.Signature):
    """Draft a top-level section of an article."""

    topic: str = dspy.InputField()
    section_heading: str = dspy.InputField()
    section_subheadings: list[str] = dspy.InputField()
    content: str = dspy.OutputField(desc="markdown-formatted section")

class DraftArticle(dspy.Module):
    def __init__(self):
        self.build_outline = dspy.ChainOfThought(Outline)
        self.draft_section = dspy.ChainOfThought(DraftSection)

    def forward(self, topic):
        outline = self.build_outline(topic=topic)
        sections = []
        for heading, subheadings in outline.section_subheadings.items():
            section, subheadings = f"## {heading}", [f"### {subheading}" for subheading in subheadings]
            section = self.draft_section(topic=outline.title, section_heading=section, section_subheadings=subheadings)
            sections.append(section.content)
        return dspy.Prediction(title=outline.title, sections=sections)

draft_article = DraftArticle()
article = draft_article(topic="World Cup 2002")

In [8]:
article

Prediction(
    title='A Comprehensive Overview of the 2002 FIFA World Cup',
    sections=["# Introduction and Tournament Background\n\nThe 2002 FIFA World Cup, held in Korea and Japan, represented a pivotal moment in the history of the tournament. It was the first time the World Cup was co-hosted by two Asian nations, marking a significant shift in the competition's global reach and popularity. Prior to the tournament, expectations were high, fueled by the strong performances of several Asian football teams in the years leading up to the event. The decision to host the World Cup in Asia was driven by a desire to expand the sport's fanbase and demonstrate Asia's growing footballing capabilities.  The tournament’s format also saw some changes, including the expansion of the number of teams participating, further solidifying its position as the world’s premier football competition.", '# Host Nation: South Korea and Japan\n\nThe 2002 FIFA World Cup was a landmark event, and a significant 